# A2 Qwen3 sentence selection v3 — Colab GPU
1054 train, 84 validation, 56 development challenge. Tiếp tục adapter RAFT v2, không phải model nền ngẫu nhiên.
Đổi Runtime → Change runtime type → T4 GPU. Tải a2_sentence_v3_bundle.zip. Không cần Modal.
Challenge không dùng chọn checkpoint và có thể từng xuất hiện trong kho train lịch sử.
Notebook chỉ được kiểm tra CPU/token khi chuẩn bị; chưa chạy GPU.

In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import json, hashlib, zipfile, shutil
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/DentalA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = DRIVE_ROOT / 'a2_sentence_v3_bundle.zip'
if not BUNDLE.exists():
    local = Path('/content/a2_sentence_v3_bundle.zip')
    if not local.exists():
        uploaded = files.upload()
        if set(uploaded) != {'a2_sentence_v3_bundle.zip'}:
            raise ValueError('Chọn đúng a2_sentence_v3_bundle.zip')
        local.write_bytes(uploaded['a2_sentence_v3_bundle.zip'])
        del uploaded
    shutil.copyfile(local, BUNDLE)
WORK = Path('/content/dental_a2_sentence_v3')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    manifest = json.loads(z.read('bundle_manifest.json'))
    assert set(z.namelist()) == set(manifest['files_sha256']) | {'bundle_manifest.json'}
    for name, digest in manifest['files_sha256'].items():
        assert not PurePosixPath(name).is_absolute() and '..' not in PurePosixPath(name).parts
        assert hashlib.sha256(z.read(name)).hexdigest() == digest, name
    z.extractall(WORK)
print('Đã giải nén:', WORK)

In [ ]:
%pip install -q -r /content/dental_a2_sentence_v3/requirements-colab-a2.txt

In [ ]:
import os, sys, subprocess
RUN_NAME = 'a2_sentence_v3_colab_01'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
env = dict(os.environ, PYTHONPATH=str(WORK), HF_HOME='/content/hf_cache_dental',
           TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')
def run_stage(stage):
    command = [sys.executable, '-u', '-X', 'utf8', '-m', 'scripts.train_a2_sentence_colab',
               '--stage', stage, '--output-dir', str(RUN_DIR)]
    with subprocess.Popen(command, cwd=WORK, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'Stage {stage} failed; xem log phía trên.')
subprocess.run(['nvidia-smi'], check=True)
print('Kết quả:', RUN_DIR)

## Kiểm tra token, không nạp trọng số

In [ ]:
run_stage('preflight')

## Validation trước train

In [ ]:
run_stage('before')

## Challenge trước train

In [ ]:
run_stage('challenge_before')

## Huấn luyện

Giữ RUN_NAME khi resume. Kết nối lại GPU và chạy lại setup/preflight rồi cell train; chỉ nhận checkpoint hoàn chỉnh trên Drive. Không đổi tên để vô tình tạo lượt mới.

In [ ]:
run_stage('train')

## Validation sau train và biểu đồ PNG

In [ ]:
run_stage('after')

## Challenge sau train

In [ ]:
run_stage('challenge_after')

## Đóng gói kết quả
Adapter ở adapter/, biểu đồ ở figures/. Metrics khớp mã câu không thay thế chấm đúng nội dung.

In [ ]:
run_stage('export')
RESULT_ZIP = RUN_DIR.parent / (RUN_NAME + '_results.zip')
files.download(str(RESULT_ZIP))